# Lecture: Fine-tuning a Language Model (SFT vs LoRA vs QLoRA)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain full fine-tuning vs SFT in one sentence each: same knobs, but SFT grades answer tokens only (§4).
- Trace LoRA's trick on 2×2 numbers: frozen `W` plus a learned nudge `B@A`, and why rank decides capacity (§5).
- Explain QLoRA's memory win: the frozen base shrinks to 0.5 bytes/knob while adapters stay trainable (§6).
- Read exact-match scores and pick the cheapest method that solves the task (§7–§8).


## The big idea

Pretraining taught the model the general rhythm of text; fine-tuning teaches it one new chore — here, writing words backwards (“king” → “gnik”). **Full fine-tuning** retrains every knob on every token. **SFT** (supervised fine-tuning) retrains every knob but grades only the *answer* tokens, masking the prompt with −100 (“ignore me”) so each step teaches the new skill, not the old prompt. **LoRA** freezes the whole base and trains only two tiny matrices per layer whose product nudges each frozen weight — few knobs, little optimizer memory. **QLoRA** squeezes the frozen base into 4-bit luggage first (0.5 bytes/knob), so the same adapters fit in ~10x less memory. Analogy: instead of rebuilding the library (full-FT), pencil notes in the margins (LoRA) and store the books in compact boxes (QLoRA).


In [ ]:
import numpy as np  # bring in numpy for the method arrays
import matplotlib.pyplot as plt  # bring in pyplot for the dashboard sketch
np.random.seed(14)  # fix the seed so the lecture is identical every run
methods = ['full', 'SFT', 'LoRA-r16', 'QLoRA-r16']  # the four lab methods, cheapest last
em = np.array([1.0, 1.0, 1.0, 1.0])  # val exact-match: every method solves reversal here
mem = np.array([7.37, 7.37, 2.37, 0.75])  # training MB: full price vs adapter price
print('EM:', em.tolist())  # show quality ties at 1.0: all four learn it
print('train MB:', mem.tolist())  # show memory falling 7.37 -> 0.75 (~10x)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: quality tie + cost ranking
ax[0].bar(methods, em, color='tab:green')  # left: everyone reaches the top
ax[0].set_ylim(0, 1.2)  # headroom so the 1.0 bars are fully visible
ax[0].set_title('Quality ties: EM 1.0 everywhere')  # title states the tie
ax[0].set_ylabel('val exact-match (higher is better)')  # label defines EM simply
ax[0].tick_params(axis='x', rotation=15)  # tilt labels so they fit
ax[1].bar(methods, mem, color=['tab:red', 'tab:red', 'tab:blue', 'tab:green'])  # right: adapters win
ax[1].set_title('Memory: 7.37 MB -> 0.75 MB (~10x)')  # title in printed numbers
ax[1].set_ylabel('train MB (lower is cheaper)')  # label defines the cost direction
ax[1].tick_params(axis='x', rotation=15)  # tilt labels so they fit
fig.tight_layout()  # stop titles and labels from overlapping
plt.show()  # display the figure inside the notebook
print('Lesson: same perfect score, ~10x less memory with QLoRA.')  # one-sentence takeaway
print('RECALIBRATE: constants above are the author CPU run (SEED=14); paste your results/finetune.csv + lorarank.csv row to replace them - ordering/shape is the claim, digits are not')


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 grades smarter, §5–§6 train cheaper, §7 scores strictly.

| Term | One-liner | Section |
|---|---|---|
| Full fine-tuning | Update every knob on every token; best signal per step, most memory | §4 |
| SFT | Full update, but loss ignores prompt tokens (masked with −100) | §4 |
| LoRA | Freeze the base, train tiny `B@A` nudges per layer; `B` starts at zero | §5 |
| Rank (r) | Inner width of the adapters; r16 solves reversal, r1 cannot | §5 |
| QLoRA | LoRA plus the frozen base squeezed to 4-bit (0.5 bytes/knob) | §6 |
| Exact-match (EM) | Share of answers reproduced letter-for-letter; 1.0 is perfect | §7 |


### §4 Full fine-tuning and SFT — grade only the answer

**In one sentence: full fine-tuning updates every knob on every token, while SFT updates every knob but computes the loss only on the answer tokens — prompt tokens are masked with −100, which is PyTorch for “ignore me”.**

Each lab example is 16 tokens: a 12-token prompt (“reverse king is ”) plus the 4-token answer (“gnik”). Full-FT averages the loss over all 16, so most gradient teaches text the model already knows. SFT averages over just the 4 answer tokens, so every drop of gradient teaches the new skill. Watch it: the strip shows 12 gray prompt slots vs 4 green graded slots, and on toy per-token losses the full mean is 1.64 while the answer-only mean is 0.53 — same step, clearer lesson.


In [ ]:
import numpy as np  # reuse numpy for the toy per-token losses
import matplotlib.pyplot as plt  # reuse pyplot for the answer-mask diagram
np.random.seed(14)  # fix the seed for reproducibility
pos = np.array([2.1, 1.9, 2.3, 2.0, 1.8, 2.2, 2.0, 1.9, 2.1, 2.0, 1.7, 2.2, 0.9, 0.4, 0.2, 0.6])  # toy loss per token: 12 prompt + 4 answer
full = round(float(pos.mean()), 2)  # full-FT grades all 16 tokens
ans = round(float(pos[12:].mean()), 2)  # SFT grades only the last 4 (prompt masked)
print('full-FT loss (16 toks):', full)  # print 1.64: prompt noise dilutes the signal
print('SFT loss (4 answer toks):', ans)  # print 0.53: pure skill signal
fig, ax = plt.subplots(figsize=(8, 2.5))  # wide flat figure for the token strip
ax.set_xlim(0, 16)  # fix range: one slot per token
ax.set_ylim(0, 3)  # fix range for strip + labels
ax.axis('off')  # hide axes: this is a diagram, not a plot
for i in range(16):  # draw one slot per token
    col = 'lightgreen' if i >= 12 else 'lightgray'  # green = graded answer, gray = masked prompt
    ax.add_patch(plt.Rectangle((i + 0.1, 1.0), 0.8, 1.0, facecolor=col, edgecolor='k'))  # draw the slot box
ax.text(6, 0.6, 'prompt x12: masked (-100, ignored)', ha='center', fontsize=9)  # label the gray zone
ax.text(14, 0.6, 'answer x4: graded', ha='center', fontsize=9)  # label the green zone
ax.set_title('SFT: full 1.64 vs answer-only 0.53 (same step, clearer lesson)')  # title in printed numbers
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

### §5 LoRA and rank — sticky notes whose product nudges the weights

**In one sentence: LoRA freezes every base weight `W` and learns only two small matrices per layer whose product `B@A` is added to `W` — and the rank `r` (their inner width) decides how expressive the nudge can be.**

Two details make it safe. `B` starts at all zeros, so on day zero `B@A = 0` and the model behaves exactly like the base — nothing can explode at init. And the **rank** is the bottleneck: with r=1 the nudge is one outer product (all rows proportional), too rigid to express a position shuffle like reversal; the lab sweeps r=1/4/16 and only r16 solves it (val loss 2.66 → 2.37 → 0.47). Watch it: the 2×2 worked math (frozen `W` plus learned `B@A`), then the rank-vs-loss curve.


In [ ]:
import numpy as np  # reuse numpy for the tiny adapter math
import matplotlib.pyplot as plt  # reuse pyplot for the rank curve
np.random.seed(14)  # fix the seed so the toy weights match each run
W = np.array([[1.0, 2.0], [3.0, 4.0]])  # toy frozen weight: never changes during tuning
A = np.array([[0.5, -0.5]])  # toy adapter A: rank-1 row compressing the input
B = np.array([[1.0], [2.0]])  # toy adapter B: starts at 0 in the lab, nonzero here
dW = B @ A  # adapter nudge = B times A, added to the frozen weight
print('W =', W.tolist())  # show the frozen base matrix
print('dW =', dW.tolist())  # show the tiny learned nudge [[0.5,-0.5],[1.0,-1.0]]
print('W+dW =', (W + dW).tolist())  # show the effective weight actually used
ranks = [1, 4, 16]  # adapter ranks swept in the lab notebook
vloss = [2.66, 2.37, 0.47]  # measured val loss per rank: only r16 solves reversal
print('rank sweep:', list(zip(ranks, vloss)))  # show capacity must cross a threshold
fig, ax = plt.subplots(figsize=(6, 4))  # make a figure for the rank curve
ax.plot(ranks, vloss, marker='o')  # line plot of capacity vs quality
ax.set_xscale('log', base=2)  # log2 x-axis: ranks double each step
ax.set_xlabel('LoRA rank r (bigger sticky notes)')  # label defines rank simply
ax.set_ylabel('val loss (lower is better)')  # label defines the loss direction
ax.set_title('Rank must be big enough (2.66 -> 0.47)')  # title in printed numbers
ax.grid(True, alpha=0.3)  # add faint grid to read values easily
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the rank curve

### §6 QLoRA — squeeze the frozen base to 4-bit

**In one sentence: QLoRA keeps LoRA's trainable adapters in full precision but stores the frozen base at 4-bit (0.5 bytes/knob), so the base costs 8x less memory while the learned notes stay exact.**

Training memory has three tenants: the weights, their gradients, and the optimizer states (Adam keeps two extra numbers per *trainable* knob). LoRA already evicts most tenants — only 32,768 adapter knobs (7.1%) need gradients and optimizer states instead of all 460,416. QLoRA goes further and shrinks the frozen landlord: the base drops from 4 bytes/knob (1.84 MB) to 0.5 (0.23 MB). Total: full 7.37 MB → LoRA 2.37 → QLoRA 0.75, the same perfect EM for ~10x less memory. Watch it: the stacked bars with every tenant printed.


In [ ]:
import numpy as np  # reuse numpy for the memory stack arrays
import matplotlib.pyplot as plt  # reuse pyplot for the stacked bars
np.random.seed(14)  # fix the seed for reproducibility
P, A_ = 460416, 32768  # base knobs and LoRA-r16 adapter knobs (7.1%)
base_fp32 = round(P * 4 / 1e6, 2)  # frozen base at 4 bytes/knob
base_int4 = round(P * 0.5 / 1e6, 2)  # frozen base squeezed to 0.5 bytes/knob
adapt = round(A_ * 16 / 1e6, 2)  # adapters + grads + Adam states at ~16 bytes/knob
full = round(P * 16 / 1e6, 2)  # full-FT trains everything at ~16 bytes/knob
print('base fp32 MB:', base_fp32)  # print 1.84: the frozen landlord at full price
print('base int4 MB:', base_int4)  # print 0.23: the same landlord squeezed 8x
print('adapters MB:', adapt)  # print 0.52: the only trainable tenants
lora_tot = round((P * 4 + A_ * 16) / 1e6, 2)  # LoRA total from unrounded parts: 2.37
qlora_tot = round((P * 0.5 + A_ * 16) / 1e6, 2)  # QLoRA total from unrounded parts: 0.75
print('full MB:', full, '| LoRA MB:', lora_tot, '| QLoRA MB:', qlora_tot)  # print 7.37, 2.37, 0.75
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the stacks
ax.bar(['full', 'LoRA', 'QLoRA'], [full, base_fp32, base_int4], label='frozen base')  # bottom: the base cost
ax.bar(['full', 'LoRA', 'QLoRA'], [0, adapt, adapt], bottom=[full, base_fp32, base_int4], label='trainable + optimizer')  # top: trainable tenants
ax.set_ylabel('training MB (lower is cheaper)')  # label defines the cost direction
ax.set_title('QLoRA: 7.37 -> 0.75 MB (~10x)')  # title in printed numbers
ax.legend(fontsize=8)  # show which color is which tenant
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the stacks

### §7 Exact-match — the strictest possible score

**In one sentence: exact-match (EM) is 1 per answer if the prediction equals the reference letter-for-letter (after lowercasing/trimming), else 0 — no partial credit.**

Reversal is a cruel task for soft grading: “tca” for “tac” is *wrong*, full stop — one swapped letter is a failed chore. So the lab's headline metric is EM averaged over held-out words: 1.0 means every word reversed perfectly. Watch it: four toy answers where two tiny slips (“tca”, “thign”) drag EM to 0.50 — strict, but honest for this task.


In [ ]:
import numpy as np  # reuse numpy for the score array
import matplotlib.pyplot as plt  # reuse pyplot for the match/mismatch bars
np.random.seed(14)  # fix the seed for reproducibility
refs = ['gnik', 'tac', 'evol', 'night']  # correct reversals (the answer key)
preds = ['gnik', 'tca', 'evol', 'thign']  # two perfect, two one-letter slips
em = [int(p == r) for p, r in zip(preds, refs)]  # strict 1-or-0 per answer
print('per-answer EM:', em)  # print [1, 0, 1, 0]: slips get zero credit
print('mean EM:', round(float(np.mean(em)), 2))  # print 0.50: half the chore done
fig, ax = plt.subplots(figsize=(6, 3.5))  # make one figure for the bars
ax.bar(['gnik=gnik', 'tca~tac', 'evol=evol', 'thign~night'], em, color=['tab:green', 'tab:red', 'tab:green', 'tab:red'])  # green = match, red = slip
ax.set_ylim(0, 1.2)  # headroom so the 1.0 bars are fully visible
ax.set_ylabel('exact-match (1 or 0)')  # label defines the strictness
ax.set_title('EM: tiny slips score 0 (mean 0.50)')  # title in printed numbers
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the bars

## §8 How it all fits together — the system map

One comparison is a bracket with every § term playing a station. The **base** checkpoint (briefly pretrained, frozen for adapters) feeds four **methods**: full-FT and **SFT** (all knobs; §4's mask is SFT's only difference), **LoRA** (frozen base + `B@A`, §5), **QLoRA** (4-bit base + `B@A`, §6). Each trains on the same reversal pairs, then the same harness reports **trainable count / memory / time / quality** — quality graded by **EM** (§7). The map below pins each term to its station; the trace prints every method's trainable count.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
import random  # bring in stdlib random only to satisfy the seed habit (unused here)
random.seed(14)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right bracket
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
ax.text(1.5, 2.0, 'base\n(frozen)', ha='center', va='center', fontsize=8,  # left: shared start
        bbox=dict(boxstyle='round', facecolor='wheat'))  # wheat = shared checkpoint
lanes = [('full-FT\n§4', 3.2), ('SFT\n§4 mask', 2.4), ('LoRA\n§5 B@A', 1.6), ('QLoRA\n§6 4-bit', 0.8)]  # four method lanes + heights
for name, y in lanes:  # draw each method lane
    ax.text(5, y, name, ha='center', va='center', fontsize=8,  # centered method label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = method choice
    ax.annotate('', xy=(4.0, y), xytext=(2.4, 2.0), arrowprops=dict(arrowstyle='->'))  # arrow from base to lane
    ax.annotate('', xy=(7.8, 2.0), xytext=(6.0, y), arrowprops=dict(arrowstyle='->'))  # arrow from lane to scoreboard
ax.text(9.5, 2.0, 'trainable / memory /\ntime / quality (§7 EM)', ha='center', va='center', fontsize=8,  # right: shared scoreboard
        bbox=dict(boxstyle='round', facecolor='lightgreen'))  # green = scored results
ax.set_title('System map: one base, four methods, one scoreboard')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the share arithmetic
np.random.seed(14)  # fix the seed for reproducibility
P = 460416  # base knobs (lab TinyGPT 2-block width-64)
lora = 2 * 16 * 64 * 16  # adapters: 2 blocks x 16 d-r terms x d=64 x r=16
print('full trainable:', P)  # print 460416: every knob moves
print('SFT trainable:', P)  # print 460416: mask changes grading, not knob count
print('LoRA-r16 trainable:', lora)  # print 32768: only the sticky notes move
print('QLoRA-r16 trainable:', lora)  # print 32768: squeezing the base adds zero knobs
print('share:', round(100 * lora / P, 1), '%')  # print 7.1%: adapters are a thin slice

## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the bracket above, on a real reversal task:

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `SEED = 14` habit as this lecture.
2. **Cell 2 (base corpus)** — brief Shakespeare pretraining data (or synthetic fallback) — the “cook” before the new chore.
3. **Cell 3 (model + LoRA + 4-bit)** — `inject_lora` (A/B notes, B starts at zero, §5) plus the fake int4 round-trip (§6); our 2×2 is that math on toy numbers.
4. **Cell 4 (tune 4 ways + eval)** — full-FT vs SFT with the −100 answer mask (§4) vs LoRA vs QLoRA, scored by **EM** (§7) into `results/finetune.csv`.
5. **Cell 5 (rank sweep + figure)** — retries r=1/4/16 (§5: val 2.66 → 2.37 → 0.47) into `results/lorarank.csv` and `results/finetune.png`.
6. **Cell 6 (cleanup)** — suggests higher rank, unfreezing the head, or real NF4 on the DGX Spark.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Same knobs, same steps — why does SFT's answer-only loss (0.53) beat full-FT's (1.64)?**

*Answer: the numbers measure different things (§4). Full-FT's 1.64 averages 12 already-known prompt tokens with 4 new answer tokens; SFT's 0.53 grades only the 4 that teach the skill. Less dilution, clearer gradient — trace the strip in the §8 map.*

2. **LoRA trains 32,768 knobs (7.1%) yet needs more steps. Why?**

*Answer: B starts at zero (§5), so day-zero change is exactly 0 and learning warms up slowly through a thin bottleneck. The payoff is memory: 2.37 MB vs 7.37 (§6), and QLoRA's 0.75 MB by squeezing the frozen base.*

3. **Rank 1 fails (val 2.66) but rank 16 solves it (0.47). Why?**

*Answer: rank is the sticky note's width (§5). Reversing positions needs many independent directions; r=1 allows one outer product (all rows proportional) and cannot express the shuffle, while r=16 can. Bigger notes, same frozen library.*
